# token 缓冲记忆（Token Buffer Memory）

> **把对话历史裁剪到严格的 token 预算以内。用真正的分词器（tokenizer）计数，因为消息条数无法准确反映上下文窗口的占用。**

想象一个限重的行李箱。你不数物品件数，而是逐件称重，把最旧的先拿出来，直到全部装得下。token 缓冲记忆的做法一样：它按 token（而不是消息条数）度量对话历史，不断删掉最旧的消息，直到总量降到预算以内。

滑动窗口记忆（技术 02）按固定条数丢消息。但一条带代码块的消息可能消耗几千个 token，而一句简短的 "yes" 只消耗一个。token 缓冲记忆用精确计量取代这种粗放的计数。**分词器（tokenizer）** 是把文本切分成 **token**（模型内部处理的词片段）的工具。接入 `tiktoken` 这样的库后，系统在历史送入模型之前，就能准确知道它消耗多少 token。

这种精确性在生产环境里最为关键。提示词过长，系统要么悄悄截断它，要么直接报错。提示词过短，又浪费昂贵的上下文窗口空间。token 缓冲记忆同时消除这两种故障：它设置一个硬上限（`max_token_limit`），总量一旦超出，就淘汰最旧的消息。

这项技术刻意保持简单：不摘要、不嵌入（文本的向量表示）、不检索。简单正是它的优势：结果确定、速度快、逻辑清晰。需要更强的保留能力时，再叠加摘要记忆或检索增强记忆。token 缓冲记忆始终是那道守住提示词边界的底层护栏。

**读完你将理解：**
- 分词器如何度量真实的上下文占用，为什么 token 计数优于消息计数。
- 如何用 `tiktoken` 和 OpenAI SDK 从零构建一个感知 token 的记忆。
- token 预算何时够用，何时需要更复杂的方案。

## 关键概念

- **token**：语言模型读取的最小单位，大约对应一个词或词片段。单词 "unhappiness" 可能切成两个 token："un" 和 "happiness"。
- **分词器（`tiktoken`）**：针对特定模型把文本切分成 token 的库。OpenAI 的 `tiktoken` 支持 GPT-4、GPT-4o 及相关模型。
- **编码（encoding）**：模型使用的具体 token 词表。GPT-4 用 `cl100k_base`，GPT-4o 用 `o200k_base`。用错编码，计数就不准。
- **max_token_limit**：对话历史允许的总 token 数硬上限。总量一旦超过这个数，系统就淘汰旧消息，直到总量降到上限以内。
- **单条消息开销（per-message overhead）**：每条聊天消息除了内容本身，还带格式 token：角色标记、内容分隔符和消息结束符。通常每条消息 4 个 token。
- **消息级淘汰（message-level eviction）**：删除整条消息，不做部分裁剪。这保住了消息的完整性，但预算可能用不满。
- **上下文窗口（context window）**：模型一次调用能处理的最大 token 数。GPT-4o 支持 128K 个 token。历史预算必须给系统提示词和模型回复留出空间。

## 架构

<p align="center">
  <img src="../../images/diagrams/05_token_buffer_memory.svg" alt="token 缓冲记忆架构图" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
    NewMsg["New Message"] --> Tokenizer["Tokenizer\n(tiktoken)"]
    Tokenizer --> Counter["Token Counter\n(sum all messages)"]
    Counter --> BudgetCheck{"Total > \nmax_token_limit?"}
    BudgetCheck -- No --> TrimmedHistory["Trimmed History"]
    BudgetCheck -- Yes --> Evict["Evict Oldest\nMessage"]
    Evict --> Counter
    TrimmedHistory --> LLM["LLM"]
    LLM --> Response["Response"]
    Response --> NewMsg
```

</details>

**数据流：** 每条新消息都会经过分词。计数器对历史中所有消息求和。总量一旦超过 `max_token_limit`，就淘汰最旧的消息，并在循环里重新计数。降到预算以内后，裁剪后的历史送入 LLM。再把响应追加进历史，循环继续。

## 环境准备

安装依赖并配置 API 访问。

In [ ]:
%pip install -q tiktoken python-dotenv deepagents langchain-openai

导入 `tiktoken` 做 token 计数，导入 OpenAI SDK 调用 LLM。API key 从 `.env` 文件加载。

In [ ]:
import os
import tiktoken

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


## 实现

我们来构建一个 `TokenBufferMemory` 类，它：
1. 用 `tiktoken` 统计每条消息的 token 数。
2. 维护所有已存消息的 token 总量。
3. 总量超过 `max_token_limit` 时，淘汰最旧的消息。
4. 每次调用时，把裁剪后的历史发给 OpenAI API。

### Token 计数

准确的 token 计数是这项技术的基础。每条聊天消息消耗的 token 比内容本身多：Chat Completions API 会给每条消息包上特殊 token，包括角色标记、内容分隔符和消息结束符。这些开销通常为每条消息 4 个 token。

下面看看 `tiktoken` 如何统计不同长度消息的 token。

In [ ]:
def count_message_tokens(message: dict) -> int:
    """统计一条聊天消息的 token 数，含单条消息开销。

    开销（每条消息 4 个 token）：
      <|im_start|> + 角色 + 内容分隔符 + <|im_end|>
    """
    overhead = 4
    return overhead + len(ENCODING.encode(message["content"]))


# 演示：统计不同长度消息的 token 数
sample_messages = [
    {"role": "user", "content": "yes"},
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "user", "content": (
        "Write a Python function that computes the nth Fibonacci number "
        "using dynamic programming with memoization."
    )},
]

for msg in sample_messages:
    tokens = count_message_tokens(msg)
    print(f"  {tokens:3d} tokens | \"{msg['content'][:70]}\"")

### TokenBufferMemory 类

这个类把 token 计数封装成一套完整的记忆系统。每次调用 `chat()` 时，它会：
1. 追加用户消息。
2. 对所有已存消息的 token 求和。
3. 循环淘汰最旧的消息，直到总量降到 `max_token_limit` 以内。
4. 把裁剪后的历史发给 LLM。
5. 追加模型响应。

把 `max_token_limit` 设为历史可用的空间：用模型的上下文窗口，减去系统提示词的 token 和回复预算。例如，128K 的窗口、200 token 的系统提示词、1,024 token 的回复预算，`max_token_limit` 设 126,000 左右即可。

In [ ]:
class TokenBufferMemory:
    """带严格 token 预算的对话记忆。"""

    def __init__(
        self,
        model,
        max_token_limit: int = 2048,
        system_prompt: str | None = None,
    ):
        # deepagents 智能体本身无状态，token 预算逻辑留在本类里
        self.agent = create_deep_agent(model=model, system_prompt=system_prompt)
        self.model = model
        self.max_token_limit = max_token_limit
        self.system_prompt = system_prompt

        # 用 cl100k_base 做预算估算；任何 OpenAI 兼容端点都适用
        self.encoding = tiktoken.get_encoding("cl100k_base")

        # 核心数据结构：消息缓冲区
        self.messages: list[dict] = []

        # 记录淘汰情况，便于检查
        self.eviction_log: list[dict] = []

    # -- Token 计数 ---------------------------------------------------
    def count_tokens(self, message: dict) -> int:
        """统计一条消息的 token 数，含单条消息开销。"""
        overhead = 4  # <|im_start|>、角色、分隔符、<|im_end|>
        return overhead + len(self.encoding.encode(message["content"]))

    def total_tokens(self) -> int:
        """对所有已存消息的 token 数求和。"""
        return sum(self.count_tokens(m) for m in self.messages)


现在添加淘汰循环和 `chat` 方法。`_trim` 方法运行一个 while 循环：弹出最旧的消息，减去它的 token 数，重复执行，直到总量降到 `max_token_limit` 以内。`chat` 方法追加用户消息，按需裁剪，调用 LLM，再追加响应。

In [ ]:
# -- 淘汰循环 ------------------------------------------------------
def _trim(self) -> None:
    """不断删除最旧的消息，直到 token 总量降到预算以内。"""
    total = self.total_tokens()
    while total > self.max_token_limit and len(self.messages) > 1:
        removed = self.messages.pop(0)
        removed_tokens = self.count_tokens(removed)
        total -= removed_tokens
        self.eviction_log.append({
            "role": removed["role"],
            "preview": removed["content"][:60],
            "tokens_freed": removed_tokens,
        })


# -- 对话 ----------------------------------------------------------
def _chat(self, user_input: str) -> str:
    """发送消息，超预算则先裁剪，再返回响应。"""
    self.messages.append({"role": "user", "content": user_input})
    self._trim()

    result = self.agent.invoke({"messages": list(self.messages)})
    assistant_text = to_text(result["messages"][-1].content)

    self.messages.append({"role": "assistant", "content": assistant_text})
    return assistant_text


TokenBufferMemory._trim = _trim
TokenBufferMemory.chat = _chat

print("✓ TokenBufferMemory methods attached")


这里是检查和重置记忆的工具方法。`get_messages` 返回当前历史的副本。`clear` 清空消息和淘汰日志。

In [ ]:
# -- 工具方法 ------------------------------------------------------
def _get_messages(self) -> list[dict]:
    """返回消息历史的副本。"""
    return [m.copy() for m in self.messages]


def _clear(self) -> None:
    """清空所有消息和日志。"""
    self.messages.clear()
    self.eviction_log.clear()


def _len(self) -> int:
    return len(self.messages)


def _repr(self) -> str:
    return (
        f"TokenBufferMemory("
        f"{len(self.messages)} msgs, "
        f"{self.total_tokens()}/{self.max_token_limit} tokens)"
    )


TokenBufferMemory.get_messages = _get_messages
TokenBufferMemory.clear = _clear
TokenBufferMemory.__len__ = _len
TokenBufferMemory.__repr__ = _repr

print("✓ TokenBufferMemory helper methods attached")


## 运行示例

一段简短对话展示 token 缓冲记忆的实际效果。预算宽裕（2,048 token）时，所有消息都装得下，不触发任何淘汰。智能体记得一切。

In [ ]:
memory = TokenBufferMemory(
    model=llm,
    max_token_limit=2048,
    system_prompt="You are a helpful, concise assistant. Keep replies under 2 sentences.",
)

exchanges = [
    "Hi! My name is Alice and I'm a machine-learning engineer.",
    "I'm building a project about agent memory systems.",
    "What's my name and what am I working on?",  # 记忆测试
]

for msg in exchanges:
    print(f"User:  {msg}")
    reply = memory.chat(msg)
    print(f"Agent: {reply}")
    print(f"       [{memory.total_tokens()} / {memory.max_token_limit} tokens, "
          f"{len(memory)} messages]\n")

### 淘汰的实际效果

现在换一个极小的预算（300 token）。几轮之后就会触发淘汰。观察 token 数如何保持有界，旧消息如何逐渐消失。

In [ ]:
small_memory = TokenBufferMemory(
    model=llm,
    max_token_limit=300,  # 故意设小，以触发淘汰
    system_prompt="Reply in one short sentence.",
)

demo_messages = [
    "My name is Bob and I live in Tokyo.",
    "I have two cats named Mochi and Sushi.",
    "My favorite language is Rust.",
    "I work at a robotics startup with 20 engineers.",
    "What do you remember about me?",  # 早前的信息会丢失
]

for msg in demo_messages:
    tokens_before = small_memory.total_tokens()
    reply = small_memory.chat(msg)
    tokens_after = small_memory.total_tokens()
    print(f"User:  {msg}")
    print(f"Agent: {reply}")
    print(f"       [tokens: {tokens_before} -> {tokens_after}, "
          f"messages: {len(small_memory)}]\n")

# 展示淘汰了哪些内容
if small_memory.eviction_log:
    print("--- Eviction log ---")
    for entry in small_memory.eviction_log:
        print(f"  Removed ({entry['role']}): "
              f"\"{entry['preview']}\" "
              f"({entry['tokens_freed']} tokens freed)")

### 为什么 token 计数优于消息计数

保留最近 5 条消息的滑动窗口，把所有消息一视同仁。实际上，消息的大小可能相差几个数量级。一个词的回复和一整段粘贴的代码，在滑动窗口里都算「一条消息」，但 token 数差别巨大。

下面的单元格直接展示这一点。

In [ ]:
comparison_messages = [
    {"role": "user", "content": "yes"},
    {"role": "assistant", "content": "No problem!"},
    {"role": "user", "content": (
        "Here's my entire configuration file:\n"
        "import logging\n"
        "import os\n"
        "from pathlib import Path\n\n"
        "BASE_DIR = Path(__file__).resolve().parent\n"
        "SECRET_KEY = os.environ['SECRET_KEY']\n"
        "DEBUG = os.environ.get('DEBUG', 'False') == 'True'\n"
        "ALLOWED_HOSTS = os.environ.get('ALLOWED_HOSTS', '').split(',')\n"
        "DATABASES = {\n"
        "    'default': {\n"
        "        'ENGINE': 'django.db.backends.postgresql',\n"
        "        'NAME': os.environ['DB_NAME'],\n"
        "        'USER': os.environ['DB_USER'],\n"
        "        'PASSWORD': os.environ['DB_PASSWORD'],\n"
        "        'HOST': os.environ.get('DB_HOST', 'localhost'),\n"
        "        'PORT': os.environ.get('DB_PORT', '5432'),\n"
        "    }\n"
        "}\n"
        "LOGGING = {\n"
        "    'version': 1,\n"
        "    'handlers': {\n"
        "        'console': {'class': 'logging.StreamHandler'},\n"
        "    },\n"
        "    'root': {'handlers': ['console'], 'level': 'WARNING'},\n"
        "}\n"
    )},
]

print("Message count vs. token count:\n")
total_tokens = 0
for i, msg in enumerate(comparison_messages):
    tokens = count_message_tokens(msg)
    total_tokens += tokens
    preview = msg["content"][:50].replace("\n", " ")
    suffix = "..." if len(msg["content"]) > 50 else ""
    print(f"  Message {i + 1} ({msg['role']:>9}): {tokens:4d} tokens "
          f"| \"{preview}{suffix}\"")

print(f"\n  Total: {total_tokens} tokens across {len(comparison_messages)} messages")
print(f"\n  A 3-message sliding window keeps all three.")
print(f"  A 50-token budget would drop message 3 "
      f"({count_message_tokens(comparison_messages[2])} tokens).")
print(f"  Token-based trimming makes the right call here.")

## 权衡

### token 缓冲记忆的适用场景
- **精确的预算控制**：你确切知道有多少 token 送入模型。不会静默截断，也不会因超长而报错。
- **消息大小不一**：代码块、JSON 载荷和简短回复都能得到准确计量。消息计数裁剪做不到这一点。
- **速度快**：淘汰在毫秒级完成，不需要调用 LLM（摘要记忆则不同）。裁剪过程是确定性的。
- **多模型路由**：如果系统会路由到上下文窗口不同的模型，就为每个模型设置各自的 `max_token_limit`。同一个类就能覆盖。

### 什么时候会出问题
- **淘汰的内容无法找回**：消息一旦删除，信息就没了。摘要记忆或检索增强记忆能保留其中一部分。
- **一条大消息挤掉许多小消息**：一条 1,000 token 的消息，会迫使系统淘汰许多短消息来腾地方。
- **分词器必须与模型匹配**：用 GPT-4 的编码去数 GPT-4o（或反过来），结果就是错的。编码要和模型保持同步。
- **整条消息淘汰可能浪费预算**：预算 500 token，最旧的消息却占 400，淘汰它腾出的空间超出需要。部分裁剪更高效，但会破坏消息的完整性。

### 下一步学什么
- **[06：向量库记忆](../06_vector_store_memory/)** 用语义检索找回相关的历史消息，而不是按新旧丢弃。
- **[03：摘要记忆](../03_summary_memory/)** 把旧消息压缩成摘要，信息不会完全丢失。
- 组合使用：用 token 缓冲做硬上限，再用摘要或检索保留淘汰的内容。

## 延伸阅读

- [LangChain ConversationTokenBufferMemory](https://python.langchain.com/docs/modules/memory/types/token_buffer?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：LangChain 的内置实现，封装 tiktoken，自动按 token 裁剪。
- [tiktoken: OpenAI's Fast BPE Tokenizer](https://github.com/openai/tiktoken)：OpenAI 模型的参考分词器。
- [OpenAI Cookbook: How to Count Tokens with tiktoken](https://cookbook.openai.com/examples/how_to_count_tokens_with_tiktoken?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：精确统计 token 的分步指南，覆盖 Chat Completions API 的单条消息开销。
- [Anthropic Token Counting API](https://docs.anthropic.com/en/docs/build-with-claude/token-counting?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：Anthropic 为 Claude 模型提供的服务端 token 计数接口。

---

*← 上一篇：[04：摘要缓冲记忆](../04_summary_buffer_memory/) · 下一篇：[06：向量库记忆](../06_vector_store_memory/) →*

## 🧪 动手练习

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：基于优先级的裁剪
修改 `_trim()`，给每条消息打分（例如，含问题或人名的消息优先级更高），优先淘汰优先级最低的消息，而不是最旧的消息。跑一段 15 轮的对话，比较回忆效果与默认 FIFO（先进先出）方案的差异。

### 挑战 2：预算扫描
把 `max_token_limit` 依次设为 500、1000、2000 和 4000，跑同一段对话。对每个预算，记录保留的消息数、淘汰的消息数，以及智能体答对了多少道回忆题。用表格呈现结果。

### 挑战 3：token 预算加摘要溢出
`_trim()` 淘汰消息时，不要直接丢弃，而是交给摘要器处理，并把摘要作为系统提示词的一部分放在最前面。统计每轮消耗的总 token 数，与纯 token 缓冲对比。这把 05 token 缓冲记忆和 03 摘要记忆的方法结合了起来。

![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--05-token-buffer-memory--token-buffer-memory)
